# Tarea 2 - Pregunta 3
## Limpieza y transformación de datos

### Objetivo

En este notebook se realizará el proceso de limpieza y transformación
de los datos de contratación correspondientes a la entidad INVIAS,
tomando como punto de partida los resultados obtenidos en la auditoría
inicial.

El proceso busca preparar los datos para las etapas posteriores de
análisis estadístico y generación de resultados, conservando las
observaciones que puedan representar señales de alerta para la pregunta
de negocio.

### Criterios generales de limpieza

Las transformaciones realizadas en este notebook estarán orientadas a:

- Corregir formatos y tipos de datos.
- Estandarizar variables categóricas y de texto cuando sea necesario.
- Tratar los valores faltantes de acuerdo con el significado de cada
  variable.
- Identificar y tratar registros duplicados cuando corresponda.
- Preparar las variables financieras y temporales para el análisis.
- Conservar los casos identificados durante la auditoría que puedan ser
  relevantes para el análisis de riesgo contractual.

Las decisiones de eliminación o transformación de registros serán
documentadas y justificadas para garantizar la trazabilidad del proceso.

In [2]:
import pandas as pd

# Cargar el archivo de datos de INVIAS
df = pd.read_csv("../../invias.csv")

# Verificar las dimensiones iniciales del conjunto de datos
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 89


/var/folders/bd/ygh4x6gj0jd4vfxdxdjck_s00000gn/T/ipykernel_10173/192952563.py:4: DtypeWarning: Columns (0: direccion_de_ejecucion_del_contrato) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../invias.csv")


In [3]:
# Revisar los tipos de datos presentes en la columna problemática

df["direccion_de_ejecucion_del_contrato"].map(type).value_counts()

direccion_de_ejecucion_del_contrato
<class 'float'>    25602
<class 'str'>          3
Name: count, dtype: int64

In [4]:
# Identificar los registros que contienen texto en la columna direccion_de_ejecucion_del_contrato

registros_texto = df[
    df["direccion_de_ejecucion_del_contrato"].apply(lambda x: isinstance(x, str))
]

registros_texto[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "direccion_de_ejecucion_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,direccion_de_ejecucion_del_contrato
10409,CO1.PCCNTR.7898538,1151-2025,terminado,No definido
24503,CO1.PCCNTR.7894143,1147-2025,Cerrado,No definido
24550,CO1.PCCNTR.7898079,1152-2025,terminado,No definido


### 2.1 Tratamiento de `direccion_de_ejecucion_del_contrato`

Durante la carga inicial se identificó una advertencia de tipos mixtos en
la variable `direccion_de_ejecucion_del_contrato`.

La revisión mostró que 25.602 registros corresponden a valores faltantes
(`NaN`) y solamente 3 registros contienen texto. Los tres registros
presentan el valor `"No definido"`.

Dado que la variable no proporciona información efectiva sobre la
dirección de ejecución del contrato y presenta un nivel de ausencia de
información prácticamente total, se decide excluirla del conjunto de
variables utilizado para el análisis.

Esta decisión no implica eliminar registros, sino únicamente retirar una
variable que no aporta información analítica suficiente.

In [5]:
# Eliminar la variable sin información analítica suficiente

df = df.drop(columns=["direccion_de_ejecucion_del_contrato"])

print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 88


### 2.2 Conversión de variables de fecha

Las variables `fecha_de_firma`, `fecha_de_inicio_del_contrato` y
`fecha_de_fin_del_contrato` contienen información temporal relevante
para el análisis contractual.

Estas variables serán convertidas al formato datetime de pandas para
facilitar posteriormente el análisis por año, duración y relaciones
temporales entre los contratos.

Los valores faltantes serán conservados como valores nulos y no serán
imputados en esta etapa, dado que la ausencia de una fecha puede tener
significado para determinados estados contractuales.

In [6]:
# Convertir las variables de fecha al formato datetime

columnas_fecha = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for columna in columnas_fecha:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")

# Verificar los tipos de datos resultantes
df[columnas_fecha].dtypes

fecha_de_firma                  datetime64[us]
fecha_de_inicio_del_contrato    datetime64[us]
fecha_de_fin_del_contrato       datetime64[us]
dtype: object

### 2.3 Validación de las variables de fecha

Después de convertir las variables temporales al formato datetime, se
verifica que la transformación haya conservado la cantidad de valores
válidos y faltantes identificados durante la auditoría inicial.

In [7]:
# Verificar valores válidos y faltantes después de la conversión

for columna in columnas_fecha:
    valores_validos = df[columna].notna().sum()
    valores_faltantes = df[columna].isna().sum()

    print(f"{columna}:")
    print(f"  Valores válidos: {valores_validos}")
    print(f"  Valores faltantes: {valores_faltantes}")
    print()

fecha_de_firma:
  Valores válidos: 17477
  Valores faltantes: 8128

fecha_de_inicio_del_contrato:
  Valores válidos: 17795
  Valores faltantes: 7810

fecha_de_fin_del_contrato:
  Valores válidos: 20074
  Valores faltantes: 5531



### 2.4 Validación de coherencia temporal

Una vez convertidas las variables temporales al formato datetime, se
verifica la consistencia cronológica de las fechas contractuales.

Se revisará si existen registros en los cuales:

- La fecha de inicio sea anterior a la fecha de firma.
- La fecha de finalización sea anterior a la fecha de inicio.

Los registros con fechas faltantes no serán considerados inconsistentes
en esta validación, dado que no existe información suficiente para
establecer una relación cronológica.

In [8]:
# Validar relaciones cronológicas entre las fechas del contrato

inicio_antes_firma = df[
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
]

fin_antes_inicio = df[
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
]

print("Inicio anterior a firma:", len(inicio_antes_firma))
print("Fin anterior a inicio:", len(fin_antes_inicio))

Inicio anterior a firma: 2959
Fin anterior a inicio: 1


### 2.5 Investigación de fechas de inicio anteriores a la firma

La validación temporal identificó 2.959 contratos cuya fecha de inicio
registrada es anterior a la fecha de firma.

Debido al número significativo de casos, no se asumirán automáticamente
como errores de calidad de datos. Se realizará una revisión descriptiva
por estado del contrato para determinar si el comportamiento se concentra
en determinados estados contractuales.

Los registros serán conservados mientras se determina su tratamiento.

In [9]:
# Distribución por estado de los contratos con inicio anterior a la firma

inicio_antes_firma["estado_contrato"].value_counts()

estado_contrato
Cerrado       1156
terminado      826
Modificado     504
Aprobado       470
Suspendido       3
Name: count, dtype: int64

In [10]:
# Porcentaje de casos con inicio anterior a la firma sobre los contratos
# que tienen disponibles ambas fechas

total_con_firma_inicio = (
    df["fecha_de_firma"].notna() &
    df["fecha_de_inicio_del_contrato"].notna()
).sum()

porcentaje_inicio_antes_firma = (
    len(inicio_antes_firma) / total_con_firma_inicio * 100
)

print("Contratos con firma e inicio disponibles:", total_con_firma_inicio)
print("Inicio anterior a firma:", len(inicio_antes_firma))
print(f"Porcentaje: {porcentaje_inicio_antes_firma:.2f}%")

Contratos con firma e inicio disponibles: 17285
Inicio anterior a firma: 2959
Porcentaje: 17.12%


In [11]:
fin_antes_inicio[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato
13181,CO1.PCCNTR.8746765,2320-2025,En ejecución,2025-12-31,2026-01-05,2025-09-04


### 2.6 Creación de indicadores de inconsistencias temporales

La revisión de las fechas identificó 2.959 contratos cuya fecha de inicio
es anterior a la fecha de firma, equivalentes al 17,12 % de los contratos
con ambas fechas disponibles.

Debido a la concentración de estos casos en determinados estados
contractuales, no se consideran automáticamente errores de digitación y
se conservan las fechas originales.

Adicionalmente, se identificó un contrato cuya fecha de finalización es
anterior a la fecha de inicio. Este caso será igualmente conservado.

Para preservar la información original y facilitar el análisis posterior,
se crean variables indicadoras que permitan identificar estas
inconsistencias sin modificar las fechas de origen.

In [12]:
# Crear indicadores de inconsistencias temporales

df["inicio_antes_firma"] = (
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
)

df["fin_antes_inicio"] = (
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
)

print("Inicio anterior a firma:", df["inicio_antes_firma"].sum())
print("Fin anterior a inicio:", df["fin_antes_inicio"].sum())

Inicio anterior a firma: 2959
Fin anterior a inicio: 1
